# PodSkipper teacher on Kaggle (Pass 65)
Settings: **Accelerator GPU T4 x2**, **Internet on**. Add two inputs: a Dataset holding `kaggle_teacher.py` and `batch-input.jsonl` (from `distill_reader.py annotate batch` or `sporc_sb_distill_pipeline.py teacher-batch`), and — after the first session — a Dataset holding the previous `batch-output.jsonl`. Then **Save Version → Save & Run All (Commit)**; it runs in the background for up to 12 h. Download `batch-output.jsonl` from the version's Output.

In [ ]:
import glob, os, shutil
src = glob.glob('/kaggle/input/**/kaggle_teacher.py', recursive=True)[0]
shutil.copy(src, '/kaggle/working/kaggle_teacher.py')
BATCH = glob.glob('/kaggle/input/**/batch-input.jsonl', recursive=True)[0]
EARLIER = (glob.glob('/kaggle/input/**/batch-output.jsonl', recursive=True) or [''])[0]
print(BATCH, EARLIER or '(first session)')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp b12283 (the release PodSkipper ships) with CUDA for the T4: ~10 min
!python /kaggle/working/kaggle_teacher.py setup

In [ ]:
# The teacher: Qwen3.8 27B, Unsloth Dynamic 4-bit (~17 GB, split across both T4s)
!python /kaggle/working/kaggle_teacher.py fetch --repo unsloth/Qwen3.8-27B-GGUF --pattern '*UD-Q4_K_XL*'

In [ ]:
import subprocess, time
server = subprocess.Popen(['python', '/kaggle/working/kaggle_teacher.py', 'serve', '--parallel', '4', '--ctx', '16384'],
                          stdout=open('/kaggle/working/server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(30); print(open('/kaggle/working/server.log').read()[-2000:])

In [ ]:
# Stops starting new windows after 11.3 h so the session can save its output.
!python /kaggle/working/kaggle_teacher.py run --input "$BATCH" --output /kaggle/working/batch-output.jsonl --resume-from "$EARLIER" --parallel 4 --hours 11.3

In [ ]:
server.terminate()
!wc -l /kaggle/working/batch-output.jsonl